# Steps 7 & 8: Anomaly Segmentation Baselines

- Step 7: pixel-based baselines using ERFNet (MSP, MaxLogit, Max Entropy)
- Step 8: mask-based baselines using EoMT (MSP, MaxLogit, Max Entropy, RbA) evaluated on three checkpoints + temperature scaling

In [1]:
!pip install ood_metrics >/dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
!pip install gitignore_parser > /dev/null
!pip install lightning > /dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you

## Setup and Imports

In [2]:
import os
import sys
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from google.colab import drive

!pip install numpy==2.0.0

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/FundGitHubProject/'
for path in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'eomt')]:
    if path not in sys.path:
        sys.path.insert(0, path)

if not os.path.exists('/content/Fundamental_Project'):
    os.symlink(PROJECT_ROOT, '/content/Fundamental_Project')

os.chdir('/content/Fundamental_Project')

Mounted at /content/drive


In [3]:
# Scoring functions and metric computation from the project repo
from posthoc_metrics import (
    get_pixel_msp,
    get_pixel_max_logit,
    get_pixel_entropy,
    get_mask_msp,
    get_mask_max_logit,
    get_mask_entropy,
    get_mask_rba,
    compute_metrics,
    cache_model_outputs,
    fast_temperature_search,
)
from eval.Validation_Dataset import anomaly_datasets
from eomt.checkpoint_utils import get_finetuned_model
from eval.erfnet import ERFNet

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## Load datasets
Same five benchmarks used for both Step 7 and Step 8.
Loaded once and reused across all models and methods.


In [5]:
DATASETS = {
    'SMIYC RA-21':  'RoadAnomaly21',
    'SMIYC RO-21':  'RoadObstacle21',
    'FS L&F':       'FS_LostFound',
    'FS Static':    'FS_Static',
    'Road Anomaly': 'RoadAnomaly',
}

dataloaders = {}
for display_name, internal_name in DATASETS.items():
    dm = anomaly_datasets.AnomalyDataModule(
        dataset_name=internal_name,
        img_size=(640, 640)
    )
    dm.setup()
    dataloaders[display_name] = dm.val_dataloader()
    print(f"Loaded: {display_name}")

Loaded: SMIYC RA-21
Loaded: SMIYC RO-21
Loaded: FS L&F
Loaded: FS Static
Loaded: Road Anomaly


In [6]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def evaluate(model, dataloader, scoring_fn, desc="", input_scale=1.0, normalize=False, debug=False):
    """
    Run inference over a dataloader and compute anomaly segmentation metrics.

    Args:
        model:       the segmentation model (ERFNet or EoMT), already on device
        dataloader:  yields batches with 'image' and 'label' keys
        scoring_fn:  callable that takes model output and returns an anomaly
                     score map of shape (B, H, W). Higher = more anomalous.
        desc:        label shown in the progress bar
        input_scale: multiply images by this before passing to model (use 255.0 for ERFNet)
        normalize:   apply ImageNet mean/std normalization (use True for EoMT)
        debug:       if True, print normal vs anomaly score gap on first batch

    Returns:
        dict with 'auprc' and 'fpr95' keys
    """
    model.eval()
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc=desc, leave=False)):
            images = batch['image'].to(device) * input_scale
            labels = batch['label']              # (B, H, W), CPU, 255 = ignore

            if normalize:
                images = (images - IMAGENET_MEAN) / IMAGENET_STD

            output = model(images)
            scores = scoring_fn(output)          # (B, H, W)

            # Resize to match label resolution if needed
            if scores.shape[-2:] != labels.shape[-2:]:
                scores = torch.nn.functional.interpolate(
                    scores.unsqueeze(1),
                    size=labels.shape[-2:],
                    mode='bilinear',
                    align_corners=False,
                ).squeeze(1)

            if debug and i == 0:
                valid        = labels != 255
                normal_mask  = valid & (labels == 0)
                anomaly_mask = valid & (labels == 1)
                s = scores.cpu()
                print(f"\n[debug] score range : [{s.min():.4f}, {s.max():.4f}]")
                if normal_mask.any() and anomaly_mask.any():
                    print(f"[debug] normal  mean : {s[normal_mask].mean():.4f}")
                    print(f"[debug] anomaly mean : {s[anomaly_mask].mean():.4f}")
                    print(f"[debug] gap          : {s[anomaly_mask].mean() - s[normal_mask].mean():.4f}  (should be > 0)")
                else:
                    print("[debug] first batch has no anomaly pixels — try a different dataset for debug")

            # Exclude void/ignore pixels
            valid = labels != 255
            all_scores.append(scores.cpu().numpy()[valid.numpy()])
            all_labels.append(labels.numpy()[valid.numpy()])

    return compute_metrics(
        np.concatenate(all_scores),
        np.concatenate(all_labels),
    )


def run_evaluation(model, model_name, scoring_methods, dataloaders, input_scale=1.0, normalize=False, debug=False):
    rows = []
    for method_name, scoring_fn in scoring_methods.items():
        row = {'Model': model_name, 'Method': method_name}
        for ds_name, loader in dataloaders.items():
            metrics = evaluate(
                model, loader, scoring_fn,
                desc=f"{model_name} | {method_name} | {ds_name}",
                input_scale=input_scale,
                normalize=normalize,
                debug=(debug and ds_name == 'SMIYC RA-21' and method_name == 'MSP'),
            )
            row[f"{ds_name} AuPRC"] = round(metrics['auprc'] * 100, 2)
            row[f"{ds_name} FPR95"] = round(metrics['fpr95'] * 100, 2)
        rows.append(row)
        print(f"  {method_name} done")
    return rows

# Step 7: Pixel-based Baselines (ERFNet)
Evaluating ERFNet with pixel-wise scoring methods.

## Load ERFNet

In [7]:
# Load ERFNet
erfnet = ERFNet(num_classes=20).to(device)

weights_path = 'trained_models/erfnet_pretrained.pth'
if os.path.exists(weights_path):
    checkpoint = torch.load(weights_path, map_location=device)
    state_dict = checkpoint.get('state_dict', checkpoint)
    state_dict = {
        k[len('module.'):] if k.startswith('module.') else k: v
        for k, v in state_dict.items()
    }
    missing, unexpected = erfnet.load_state_dict(state_dict, strict=False)
    print(f"ERFNet loaded — missing: {len(missing)}, unexpected: {len(unexpected)}")

ERFNet loaded — missing: 2, unexpected: 0


In [8]:
# Scoring functions for pixel model.
# Each takes the raw model output (B, C, H, W) and returns (B, H, W).
pixel_scoring_methods = {
    'MSP':         lambda output: get_pixel_msp(output),
    'MaxLogit':    lambda output: get_pixel_max_logit(output),
    'Max Entropy': lambda output: get_pixel_entropy(output),
}

In [9]:
print("Evaluating ERFNet...")
pixel_results = run_evaluation(
    erfnet, 'ERFNet', pixel_scoring_methods, dataloaders,
    input_scale=1.0, normalize=False
)

Evaluating ERFNet...


ERFNet | MSP | SMIYC RA-21:   0%|          | 0/10 [00:00<?, ?it/s]

ERFNet | MSP | SMIYC RO-21:   0%|          | 0/30 [00:00<?, ?it/s]

ERFNet | MSP | FS L&F:   0%|          | 0/100 [00:00<?, ?it/s]

ERFNet | MSP | FS Static:   0%|          | 0/30 [00:00<?, ?it/s]

...

ERFNet | Max Entropy | SMIYC RO-21:   0%|          | 0/30 [00:00<?, ?it/s]

ERFNet | Max Entropy | FS L&F:   0%|          | 0/100 [00:00<?, ?it/s]

ERFNet | Max Entropy | FS Static:   0%|          | 0/30 [00:00<?, ?it/s]

ERFNet | Max Entropy | Road Anomaly:   0%|          | 0/60 [00:00<?, ?it/s]

  Max Entropy done


In [10]:
df_pixel = pd.DataFrame(pixel_results)
print("\nStep 7 results:")
print(df_pixel.to_string(index=False))


Step 7 results:
 Model      Method  SMIYC RA-21 AuPRC  SMIYC RA-21 FPR95  SMIYC RO-21 AuPRC  SMIYC RO-21 FPR95  FS L&F AuPRC  FS L&F FPR95  FS Static AuPRC  FS Static FPR95  Road Anomaly AuPRC  Road Anomaly FPR95
ERFNet         MSP              21.05              79.89               2.63              62.53          1.63         48.82             6.68            33.61                8.95               92.23
ERFNet    MaxLogit              27.88              77.67               4.65              37.17          3.54         42.84             9.18            30.77               11.12               81.00
ERFNet Max Entropy              22.78              80.14               3.11              62.42          2.39         48.44             8.45            33.17                9.05               92.31


In [11]:
for ds_name, loader in dataloaders.items():
    total_pixels = 0
    anomaly_pixels = 0
    for batch in loader:
        labels = batch['label']
        valid = labels != 255
        anomaly_pixels += (labels[valid] == 1).sum().item()
        total_pixels += valid.sum().item()
    prevalence = anomaly_pixels / total_pixels * 100
    print(f"{ds_name}: {prevalence:.3f}% anomaly pixels")

SMIYC RA-21: 14.809% anomaly pixels
SMIYC RO-21: 0.673% anomaly pixels
FS L&F: 0.280% anomaly pixels
FS Static: 1.375% anomaly pixels
Road Anomaly: 9.849% anomaly pixels


# Step 8: Mask-based Baselines (EoMT)
Evaluating EoMT across three checkpoints with query-based and RbA scoring.

In [12]:
def make_mask_scoring_fn(fn):
    def wrapped(output):
        mask_pred_list, class_pred_list = output      # masks first, classes second
        mask_pred  = mask_pred_list[-1]  if isinstance(mask_pred_list,  list) else mask_pred_list
        class_pred = class_pred_list[-1] if isinstance(class_pred_list, list) else class_pred_list
        # mask_pred:  (B, Q, H, W)        spatial
        # class_pred: (B, Q, num_classes)  semantic
        return fn(class_pred, mask_pred)
    return wrapped

mask_scoring_methods = {
    'MSP':         make_mask_scoring_fn(get_mask_msp),
    'MaxLogit':    make_mask_scoring_fn(get_mask_max_logit),
    'Max Entropy': make_mask_scoring_fn(get_mask_entropy),
    'RbA':         make_mask_scoring_fn(get_mask_rba),
}

In [13]:
# Three checkpoints: fine-tuned (COCO -> Cityscapes via LoRA)
CHECKPOINTS = {
    'Fine-tuned': 'checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt',
}

In [14]:
from eval.iouEval import iouEval
from eomt.models.vit import ViT
from eomt.models.eomt import EoMT
from eomt.training.mask_classification_semantic import MaskClassificationSemantic
from eomt.training.mask_classification_panoptic import MaskClassificationPanoptic

In [15]:
import torch.nn.functional as F

def _load_state_dict_into(model, ckpt_path):
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    if not any(k.startswith('network.') for k in state):
        state = {f'network.{k}': v for k, v in state.items()}

    model_sd = model.state_dict()
    for key in list(state.keys()):
        if 'pos_embed' not in key:
            continue
        if key not in model_sd:
            continue
        ckpt_shape  = state[key].shape
        model_shape = model_sd[key].shape
        if ckpt_shape == model_shape:
            continue

        N_ckpt, D   = ckpt_shape[1], ckpt_shape[2]
        N_model     = model_shape[1]
        H_c = W_c   = int(N_ckpt  ** 0.5)
        H_m = W_m   = int(N_model ** 0.5)

        pe = state[key]
        pe = pe.reshape(1, H_c, W_c, D).permute(0, 3, 1, 2)
        pe = F.interpolate(pe.float(), size=(H_m, W_m),
                           mode='bicubic', align_corners=False)
        pe = pe.permute(0, 2, 3, 1).reshape(1, N_model, D)
        state[key] = pe
        print(f'  interpolated {key}: {list(ckpt_shape)} -> {list(model_shape)}')

    model.load_state_dict(state, strict=False)
    print(f'  loaded {ckpt_path}')

    return model

def load_cs_model(ckpt_path, img_size=(1024, 1024)):
    encoder = ViT(img_size=img_size, backbone_name='vit_base_patch14_reg4_dinov2')
    network = EoMT(encoder, num_classes=19, num_q=100, num_blocks=3)
    model = MaskClassificationSemantic(
        network=network, img_size=img_size, num_classes=19, attn_mask_annealing_enabled=False
    )
    return _load_state_dict_into(model, ckpt_path).eval().to(device)

def load_coco_model(ckpt_path, img_size=(640, 640)):
    stuff_classes = list(range(80, 133))
    encoder = ViT(img_size=img_size, backbone_name='vit_base_patch14_reg4_dinov2')
    network = EoMT(encoder, num_classes=133, num_q=200, num_blocks=3)
    model = MaskClassificationPanoptic(
        network=network, img_size=img_size, num_classes=133,
        stuff_classes=stuff_classes, attn_mask_annealing_enabled=False
    )
    return _load_state_dict_into(model, ckpt_path).eval().to(device)

In [16]:
## 3. Instantiate Models and DataModule
from eomt.datasets.cityscapes_semantic import CityscapesSemantic

print('Loading EoMT-Cityscapes...')
model_cs = load_cs_model("eomt/eomt_weights/eomt_cityscapes.bin", img_size=(640, 640))

print('\nLoading EoMT-COCO...')
model_coco = load_coco_model("eomt/eomt_weights/eomt_coco.bin", img_size=(640, 640))

dm_cs = CityscapesSemantic(path="eomt/data", batch_size=1, num_workers=2, img_size=(1024, 1024))
dm_cs.setup("validate")

Loading EoMT-Cityscapes...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


  interpolated network.encoder.backbone.pos_embed: [1, 4096, 768] -> [1, 1600, 768]
  loaded eomt/eomt_weights/eomt_cityscapes.bin

Loading EoMT-COCO...


  loaded eomt/eomt_weights/eomt_coco.bin


In [17]:
# Models to evaluate and their display names for the results table
eomt_models = [
    (model_cs,   'EoMT (Cityscapes)'),
    (model_coco, 'EoMT (COCO)'),
]

In [18]:
mask_results = []

for model, model_name in eomt_models:
    rows = run_evaluation(
        model, model_name, mask_scoring_methods, dataloaders,
        input_scale=1.0, normalize=True, debug=True
    )
    mask_results.extend(rows)
    torch.cuda.empty_cache()

df_mask = pd.DataFrame(mask_results)
print("\nStep 8 results:")
print(df_mask.to_string(index=False))

EoMT (Cityscapes) | MSP | SMIYC RA-21:   0%|          | 0/10 [00:00<?, ?it/s]


[debug] score range : [0.0055, 1.0000]
[debug] normal  mean : 0.1899
[debug] anomaly mean : 0.6456
[debug] gap          : 0.4557  (should be > 0)


EoMT (Cityscapes) | MSP | SMIYC RO-21:   0%|          | 0/30 [00:00<?, ?it/s]

EoMT (Cityscapes) | MSP | FS L&F:   0%|          | 0/100 [00:00<?, ?it/s]

EoMT (Cityscapes) | MSP | FS Static:   0%|          | 0/30 [00:00<?, ?it/s]

...

EoMT (COCO) | RbA | SMIYC RO-21:   0%|          | 0/30 [00:00<?, ?it/s]

EoMT (COCO) | RbA | FS L&F:   0%|          | 0/100 [00:00<?, ?it/s]

EoMT (COCO) | RbA | FS Static:   0%|          | 0/30 [00:00<?, ?it/s]

EoMT (COCO) | RbA | Road Anomaly:   0%|          | 0/60 [00:00<?, ?it/s]

  RbA done

Step 8 results:
            Model      Method  SMIYC RA-21 AuPRC  SMIYC RA-21 FPR95  SMIYC RO-21 AuPRC  SMIYC RO-21 FPR95  FS L&F AuPRC  FS L&F FPR95  FS Static AuPRC  FS Static FPR95  Road Anomaly AuPRC  Road Anomaly FPR95
EoMT (Cityscapes)         MSP              25.46              61.12              29.35              66.56          0.70         58.85             1.60            66.94               12.24               82.63
EoMT (Cityscapes)    MaxLogit              17.62              63.80              27.06             100.00          0.79         55.61             1.12            96.84               11.46               96.38
EoMT (Cityscapes) Max Entropy              33.47              56.00              30.83              69.75          0.33         66.54             2.74            70.96               13.44               86.54
EoMT (Cityscapes)         RbA              22.72              95.26              15.11              99.81          0.72         49.75       

In [19]:
with torch.no_grad():
    batch = next(iter(dataloaders['SMIYC RA-21']))
    images = batch['image'].to(device)

    print(f"Input range: [{images.min():.3f}, {images.max():.3f}]")
    print(f"Input mean per channel: {images.mean(dim=[0,2,3])}")  # should be ~[0.485, 0.456, 0.406] if ImageNet-normalized
    print(f"Input std  per channel: {images.std(dim=[0,2,3])}")   # should be ~[0.229, 0.224, 0.225]

Input range: [0.004, 0.992]
Input mean per channel: tensor([0.4544, 0.4602, 0.4490], device='cuda:0')
Input std  per channel: tensor([0.1993, 0.1767, 0.1752], device='cuda:0')


## Temperature Scaling (Smart Trick)
Optimizing MSP calibration using cached logits.

In [20]:
# Fix a stale function name in the fast_eval_utils module if needed.
# (The original code patched the source file on disk — that is fragile.
#  A safer approach is to monkey-patch the module in memory.)
import posthoc_metrics.fast_eval_utils as _feu
if hasattr(_feu, 'get_msp_anomaly_map') and not hasattr(_feu, 'get_mask_msp'):
    _feu.get_mask_msp = _feu.get_msp_anomaly_map
    importlib.reload(sys.modules['posthoc_metrics'])

from posthoc_metrics import fast_temperature_search

In [21]:
# Step 1: run inference once and cache logits
calib_model = get_finetuned_model(CHECKPOINTS['Fine-tuned']).to(device)
calib_model.eval()

CACHE_DIR = 'temp_scaling_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# The cache function expects (image, label) tuples, not dicts.
class TupleLoader:
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for batch in self.loader:
            yield batch['image'], batch['label']
    def __len__(self):
        return len(self.loader)

cache_model_outputs(
    calib_model,
    TupleLoader(dataloaders['Road Anomaly']),
    CACHE_DIR,
    device=device,
)
del calib_model
torch.cuda.empty_cache()

--- Initializing Enhanced Architecture ---
Blocks: 3 | LoRA R: 8 | Backbone: vit_base_patch14_reg4_dinov2


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
# Step 2: sweep temperatures
# The range here is narrow (0.5 to 1.1). You could widen it if the
# optimum is at the boundary — e.g. try [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0].

import os
import numpy as np
import torch
from tqdm import tqdm

# Access the original module where fast_temperature_search is defined
import posthoc_metrics.fast_eval_utils as _feu

_original_fast_temperature_search = _feu.fast_temperature_search

def _corrected_fast_temperature_search(cache_dir, scoring_fn, temperatures):
    results = {}
    cached_files = [f for f in os.listdir(cache_dir) if f.endswith('.npz')]
    num_cached_files = len(cached_files)

    for T in temperatures:
        print(f"Testing Temperature T={T}...")
        all_scores = []
        all_gts = []

        for i in tqdm(range(num_cached_files), desc=f"Testing Temperature T={T}"):
            data = np.load(os.path.join(cache_dir, f"{i:04d}.npz"))

            # Convert to tensors and move to device
            class_preds = torch.from_numpy(data['class_preds']).to(device)
            mask_preds = torch.from_numpy(data['mask_preds']).to(device)
            labels = data['labels'] # numpy array, original GT size

            # Apply the scoring function with the current temperature
            # Assuming scoring_fn accepts temperature argument from the library's fast_temperature_search signature
            scores = scoring_fn(class_preds, mask_preds, temperature=T).cpu().numpy()

            # Ensure scores match the labels' spatial dimensions
            if scores.shape[-2:] != labels.shape[-2:]:
                scores_tensor = torch.from_numpy(scores).unsqueeze(1) # Add channel dim for interpolation
                labels_tensor = torch.from_numpy(labels) # For target size

                interpolated_scores = torch.nn.functional.interpolate(
                    scores_tensor,
                    size=labels_tensor.shape[-2:], # Use original label size for interpolation
                    mode='bilinear',
                    align_corners=False,
                ).squeeze(1).numpy() # Remove channel dim and convert back to numpy

                scores = interpolated_scores

            # Create a valid mask to filter out ignore labels (255)
            valid_mask = (labels != 255)

            # Append only valid scores and labels, flattened
            all_scores.append(scores[valid_mask].ravel())
            all_gts.append(labels[valid_mask].ravel())

        # Compute aggregate metrics for this temperature
        # `compute_metrics` is imported from `posthoc_metrics` in setup cell
        metrics = compute_metrics(np.concatenate(all_scores), np.concatenate(all_gts))
        print(f"  T={T} -> AuPRC: {metrics['auprc']:.2f} | FPR95: {metrics['fpr95']:.2f}")
        results[T] = metrics
    return results

# Monkey-patch the function in the imported module
_feu.fast_temperature_search = _corrected_fast_temperature_search

# Now, call the function using the module's reference, which is now patched.
# The fast_temperature_search function imported globally (if any) might still refer to the original.
# To be safe, let's call it via the module, or ensure the global one is updated.
# If `fast_temperature_search` was imported as `from posthoc_metrics import fast_temperature_search`,
# it might hold a reference to the old function. Reloading `posthoc_metrics` would update it.
# However, the instruction is to fix *this cell*, so directly using `_feu.fast_temperature_search` is safest.

# If `fast_temperature_search` (global) is what's expected, we need to update it:
fast_temperature_search = _feu.fast_temperature_search

temperatures = [0.5, 0.75, 1.0, 1.1]
search_results = fast_temperature_search(
    CACHE_DIR,
    scoring_fn=get_mask_msp,
    temperatures=temperatures,
)


In [ ]:
# Step 3: find best T by AuPRC
best_t = max(search_results, key=lambda t: search_results[t]['auprc'])

# Step 4: build results table
temp_rows = []
for t in temperatures:
    temp_rows.append({
        'Temperature': t,
        'AuPRC': round(search_results[t]['auprc'] * 100, 2),
        'FPR95': round(search_results[t]['fpr95'] * 100, 2),
        'Note': 'best' if t == best_t else '',
    })

In [ ]:
df_temp = pd.DataFrame(temp_rows)
print(f"\nTemperature scaling results (MSP, Fine-tuned EoMT, Road Anomaly)")
print(f"Best temperature: T = {best_t}")
print(df_temp.to_string(index=False))